# 10-class eye disease baselines — two augmentation approaches

Six ImageNet-pretrained CNN/Transformer baselines, fully fine-tuned (every parameter trainable, no freezing), each trained under two data regimes:

| Approach | Train set | Online train aug |
|---|---|---|
| **A — `offline_plus_online`** | `splits/approach_a_train.csv` (augmented files whose parent original is in train) | yes |
| **B — `online_only`** | `splits/approach_b_train.csv` (original train images) | yes (same) |

Both approaches use the **same** `splits/val.csv` and `splits/test.csv`, which contain **original images only**.
Splits come from `audit_splits_and_augmentation.py` (seed 42, per-class 70/15/15 on originals, augmented files inherit their parent's split, MD5 leakage checks). Run that script first.

Requirements: `torch`, `torchvision>=0.15`, `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `pillow`.

Outputs per run: `results/<approach>/<model>/{best.pth,last.pth,training_history.csv,metrics.json,results.txt,classification_report.txt,confusion_matrix.png,DONE.json}`.
Interrupted runs resume from `last.pth`; finished runs (`DONE.json` + `best.pth`) are skipped. The leaderboard ranks models **within each approach** by test accuracy, then test macro-F1.

In [ ]:
import gc
import json
import math
import os
import platform
import random
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 42
EPOCHS = 20
PATIENCE = 6
LR = 1e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 32
MIN_BATCH_SIZE = 2
GRAD_CLIP_NORM = 1.0
IMG_SIZE = 224

MODEL_NAMES = ["alexnet", "vgg16", "resnet50", "mobilenet_v3_large", "efficientnet_b0", "swin_t"]
APPROACHES = {
    "offline_plus_online": "approach_a_train.csv",
    "online_only": "approach_b_train.csv",
}

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"
# Notebook-defined Dataset classes cannot be unpickled by spawn-based workers (macOS/Windows).
NUM_WORKERS = 0 if platform.system() in ("Darwin", "Windows") else min(8, os.cpu_count() or 1)
PIN_MEMORY = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"torch {torch.__version__} | device={DEVICE} | amp={USE_AMP} | workers={NUM_WORKERS}")

In [ ]:
PROJECT_SPLITS = Path("/Users/mdimam/Error/ResearchWorkspace/MAVIS/EyeDiseaseClassification/ClassWorkOne/splits")


def _shallow_dirs(root: Path, max_depth: int = 5):
    if not root.is_dir():
        return
    stack = [(root, 0)]
    while stack:
        cur, depth = stack.pop()
        yield cur
        if depth >= max_depth:
            continue
        try:
            kids = [p for p in cur.iterdir() if p.is_dir() and not p.name.startswith(".")]
        except OSError:
            continue
        for k in kids:
            stack.append((k, depth + 1))


def find_splits_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "splits",
        cwd / "ClassWorkOne" / "splits",
        cwd / "EyeDiseaseClassification" / "ClassWorkOne" / "splits",
        cwd / "MAVIS" / "EyeDiseaseClassification" / "ClassWorkOne" / "splits",
        cwd.parent / "splits",
        cwd.parent / "ClassWorkOne" / "splits",
        PROJECT_SPLITS,
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for d in _shallow_dirs(kaggle_input, max_depth=5):
            if (d / "val.csv").is_file() and (d / "test.csv").is_file() and (d / "approach_a_train.csv").is_file():
                candidates.append(d)
    seen = []
    for c in candidates:
        if c in seen:
            continue
        seen.append(c)
        if (c / "val.csv").is_file() and (c / "test.csv").is_file():
            return c
    extra = ""
    if kaggle_input.is_dir():
        names = []
        for d in _shallow_dirs(kaggle_input, max_depth=2):
            names.append(str(d))
        extra = "\n/kaggle/input tree:\n" + "\n".join(names[:40])
    raise FileNotFoundError(
        "splits/ not found. Add the ClassWorkOne/splits folder as a Kaggle dataset "
        "(it must contain val.csv, test.csv, approach_a_train.csv, approach_b_train.csv).\n"
        "Tried:\n" + "\n".join(str(c) for c in seen) + extra
    )


def _has_class_dirs(folder: Path) -> bool:
    try:
        names = {c.name for c in folder.iterdir() if c.is_dir()}
    except OSError:
        return False
    return "Glaucoma" in names and "Healthy" in names


def _unwrap_same_name(folder: Path) -> Path:
    """Kaggle zips often nest Original Dataset/Original Dataset/."""
    seen = set()
    cur = folder
    while cur not in seen:
        seen.add(cur)
        if _has_class_dirs(cur):
            return cur
        nested = cur / cur.name
        if nested.is_dir():
            cur = nested
            continue
        break
    return cur


def find_named_dataset(folder_name: str) -> Path:
    """Locate 'Original Dataset' or 'Augmented Dataset' locally or under /kaggle/input."""
    cwd = Path.cwd().resolve()
    direct = [
        cwd / folder_name,
        cwd.parent / folder_name,
        PROJECT_SPLITS.parent.parent / folder_name,
    ]
    hits = [p for p in direct if p.is_dir()]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for d in _shallow_dirs(kaggle_input, max_depth=8):
            if d.name == folder_name:
                hits.append(d)
    unwrapped = []
    seen = set()
    for h in hits:
        u = _unwrap_same_name(h)
        if u not in seen:
            seen.add(u)
            unwrapped.append(u)
    with_classes = [u for u in unwrapped if _has_class_dirs(u)]
    if with_classes:
        return with_classes[0]
    if len(unwrapped) == 1:
        return unwrapped[0]
    if unwrapped:
        unwrapped.sort(key=lambda p: sum(1 for c in p.iterdir() if c.is_dir()), reverse=True)
        return unwrapped[0]
    raise FileNotFoundError(
        f"Could not find folder {folder_name!r}. On Kaggle, Add data for that zip "
        f"so it appears under /kaggle/input. cwd={cwd}"
    )


SPLITS_DIR = find_splits_dir()
ORIG_DIR = find_named_dataset("Original Dataset")
AUG_DIR = find_named_dataset("Augmented Dataset")
if Path("/kaggle/working").is_dir():
    RESULTS_DIR = Path("/kaggle/working/eye_10class_results")
else:
    RESULTS_DIR = SPLITS_DIR.parent / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def resolve_image_path(row, folder: str) -> str:
    p = Path(row["image_path"])
    if p.is_file():
        return str(p)
    base = ORIG_DIR if folder.startswith("Original") else AUG_DIR
    tried = []
    for alt in (
        base / row["class_name"] / row["image_name"],
        base / base.name / row["class_name"] / row["image_name"],
    ):
        tried.append(alt)
        if alt.is_file():
            return str(alt)
    raise FileNotFoundError("missing image: " + str(p) + " | tried " + " | ".join(str(t) for t in tried))

def load_split(csv_name: str, folder: str) -> pd.DataFrame:
    df = pd.read_csv(SPLITS_DIR / csv_name)
    df["image_path"] = [resolve_image_path(r, folder) for _, r in df.iterrows()]
    return df


val_df = load_split("val.csv", "Original Dataset")
test_df = load_split("test.csv", "Original Dataset")
train_dfs = {
    "offline_plus_online": load_split(APPROACHES["offline_plus_online"], "Augmented Dataset"),
    "online_only": load_split(APPROACHES["online_only"], "Original Dataset"),
}

CLASS_NAMES = sorted(val_df["class_name"].unique())
assert len(CLASS_NAMES) == 10, CLASS_NAMES
for name, df in [("val", val_df), ("test", test_df), *train_dfs.items()]:
    assert sorted(df["class_name"].unique()) == CLASS_NAMES, f"{name} is missing classes"
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

# Val/test must be originals and disjoint from both training sets by content.
eval_md5 = set(val_df["content_md5"]) | set(test_df["content_md5"])
assert not (set(val_df["content_md5"]) & set(test_df["content_md5"])), "val/test overlap"
for name, df in train_dfs.items():
    assert not (set(df["content_md5"]) & eval_md5), f"{name} train overlaps val/test"

print("splits:", SPLITS_DIR)
print("original:", ORIG_DIR)
print("augmented:", AUG_DIR)
print("results:", RESULTS_DIR)
summary = pd.DataFrame({
    "A_train": train_dfs["offline_plus_online"]["class_name"].value_counts(),
    "B_train": train_dfs["online_only"]["class_name"].value_counts(),
    "val": val_df["class_name"].value_counts(),
    "test": test_df["class_name"].value_counts(),
}).reindex(CLASS_NAMES).fillna(0).astype(int)
summary.loc["TOTAL"] = summary.sum()
print(summary.to_string())



In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class EyeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.paths = df["image_path"].tolist()
        self.labels = [CLASS_TO_IDX[c] for c in df["class_name"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            img = im.convert("RGB")
        return self.transform(img), self.labels[i]


def make_loaders(train_df: pd.DataFrame, batch_size: int):
    g = torch.Generator()
    g.manual_seed(SEED)
    common = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=NUM_WORKERS > 0)
    train_loader = DataLoader(EyeDataset(train_df, train_tf), batch_size=batch_size, shuffle=True,
                              drop_last=len(train_df) > batch_size, generator=g, **common)
    val_loader = DataLoader(EyeDataset(val_df, eval_tf), batch_size=batch_size, shuffle=False, **common)
    test_loader = DataLoader(EyeDataset(test_df, eval_tf), batch_size=batch_size, shuffle=False, **common)
    return train_loader, val_loader, test_loader, g

In [ ]:
def build_model(name: str) -> nn.Module:
    model = models.get_model(name, weights="DEFAULT")
    if name in ("alexnet", "vgg16"):
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, NUM_CLASSES)
    elif name == "resnet50":
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    elif name == "mobilenet_v3_large":
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, NUM_CLASSES)
    elif name == "efficientnet_b0":
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    elif name == "swin_t":
        model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
    else:
        raise ValueError(name)
    for p in model.parameters():
        p.requires_grad = True
    return model


def is_oom(e: BaseException) -> bool:
    if isinstance(e, torch.cuda.OutOfMemoryError):
        return True
    return isinstance(e, RuntimeError) and "out of memory" in str(e).lower()


def free_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()


def atomic_save(obj, path: Path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, correct, seen = 0.0, 0, 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        seen += y.size(0)
    return total_loss / seen, correct / seen


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, preds, targets = 0.0, [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits.float(), y)
        total_loss += loss.item() * y.size(0)
        preds.append(logits.argmax(1).cpu())
        targets.append(y.cpu())
    y_pred = torch.cat(preds).numpy()
    y_true = torch.cat(targets).numpy()
    return {
        "loss": total_loss / len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", labels=range(NUM_CLASSES), zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", labels=range(NUM_CLASSES), zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", labels=range(NUM_CLASSES), zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", labels=range(NUM_CLASSES), zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }


def save_confusion_matrix(cm, path: Path, title: str):
    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(cm, cmap="Blues")
    fig.colorbar(im, ax=ax, fraction=0.046)
    short = [c.replace(" [Color Fundus]", "") for c in CLASS_NAMES]
    ax.set_xticks(range(NUM_CLASSES), short, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES), short)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    thresh = cm.max() / 2 if cm.max() else 0.5
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black", fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

In [ ]:
def _train_and_test(approach: str, model_name: str, out: Path, batch_size: int) -> dict:
    seed_everything(SEED)
    model = build_model(model_name).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    start_epoch, best_f1, best_epoch, bad_epochs, stopped, history = 0, -1.0, 0, 0, False, []
    last_path, best_path = out / "last.pth", out / "best.pth"
    if last_path.is_file():
        ckpt = torch.load(last_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        start_epoch, best_f1, best_epoch = ckpt["epoch"], ckpt["best_f1"], ckpt["best_epoch"]
        bad_epochs, stopped, history = ckpt["bad_epochs"], ckpt["stopped"], ckpt["history"]
        torch.set_rng_state(ckpt["torch_rng"])
        print(f"  resumed from epoch {start_epoch} (best val macro-F1 {best_f1:.4f} @ {best_epoch})")

    train_loader, val_loader, test_loader, gen = make_loaders(train_dfs[approach], batch_size)

    for epoch in range(start_epoch + 1, EPOCHS + 1):
        if stopped:
            break
        gen.manual_seed(SEED + epoch)
        t0 = time.time()
        lr = optimizer.param_groups[0]["lr"]
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val = evaluate(model, val_loader, criterion)
        scheduler.step()
        history.append({
            "epoch": epoch, "lr": lr, "batch_size": batch_size,
            "train_loss": tr_loss, "train_acc": tr_acc,
            "val_loss": val["loss"], "val_acc": val["accuracy"], "val_macro_f1": val["macro_f1"],
            "seconds": round(time.time() - t0, 1),
        })
        improved = val["macro_f1"] > best_f1
        if improved:
            best_f1, best_epoch, bad_epochs = val["macro_f1"], epoch, 0
            atomic_save({"model": model.state_dict(), "epoch": epoch, "val_macro_f1": best_f1,
                         "model_name": model_name, "approach": approach, "class_names": CLASS_NAMES}, best_path)
        else:
            bad_epochs += 1
        stopped = bad_epochs >= PATIENCE
        atomic_save({
            "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
            "epoch": epoch, "best_f1": best_f1, "best_epoch": best_epoch, "bad_epochs": bad_epochs,
            "stopped": stopped, "history": history, "batch_size": batch_size,
            "torch_rng": torch.get_rng_state(),
        }, last_path)
        pd.DataFrame(history).to_csv(out / "training_history.csv", index=False)
        print(f"  ep {epoch:02d} lr {lr:.2e} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {val['loss']:.4f} acc {val['accuracy']:.4f} F1 {val['macro_f1']:.4f}"
              f"{' *' if improved else ''} | {history[-1]['seconds']}s")
        if stopped:
            print(f"  early stop: no val macro-F1 gain for {PATIENCE} epochs")

    best = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best["model"])
    test = evaluate(model, test_loader, criterion)
    val = evaluate(model, val_loader, criterion)

    report = classification_report(test["y_true"], test["y_pred"], labels=range(NUM_CLASSES),
                                   target_names=CLASS_NAMES, digits=4, zero_division=0)
    (out / "classification_report.txt").write_text(report)
    cm = confusion_matrix(test["y_true"], test["y_pred"], labels=range(NUM_CLASSES))
    save_confusion_matrix(cm, out / "confusion_matrix.png", f"{approach} / {model_name} — test")

    metrics = {
        "approach": approach, "model": model_name,
        "best_epoch": best_epoch, "epochs_run": len(history), "early_stopped": bool(stopped),
        "batch_size": batch_size, "train_images": len(train_dfs[approach]),
        "val_images": len(val_df), "test_images": len(test_df),
        **{f"val_{k}": float(v) for k, v in val.items() if k not in ("y_true", "y_pred")},
        **{f"test_{k}": float(v) for k, v in test.items() if k not in ("y_true", "y_pred")},
        "confusion_matrix": cm.tolist(), "class_names": CLASS_NAMES,
    }
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    (out / "results.txt").write_text(
        f"approach: {approach}\nmodel: {model_name}\n"
        f"best epoch: {best_epoch} / {len(history)} run (early stopped: {stopped})\n"
        f"train images: {len(train_dfs[approach])}  val: {len(val_df)}  test: {len(test_df)}\n\n"
        f"val  accuracy {val['accuracy']:.4f}  macro-F1 {val['macro_f1']:.4f}\n"
        f"test accuracy {test['accuracy']:.4f}  macro-F1 {test['macro_f1']:.4f}  "
        f"weighted-F1 {test['weighted_f1']:.4f}\n"
        f"test macro precision {test['macro_precision']:.4f}  macro recall {test['macro_recall']:.4f}\n\n"
        + report
    )
    (out / "DONE.json").write_text(json.dumps({"finished": time.strftime("%Y-%m-%d %H:%M:%S"),
                                               "test_accuracy": float(test["accuracy"]),
                                               "test_macro_f1": float(test["macro_f1"])}, indent=2))
    print(f"  TEST acc {test['accuracy']:.4f} | macro-F1 {test['macro_f1']:.4f} (best epoch {best_epoch})")
    del model, optimizer, train_loader, val_loader, test_loader
    free_memory()
    return metrics


def run(approach: str, model_name: str) -> dict:
    out = RESULTS_DIR / approach / model_name
    out.mkdir(parents=True, exist_ok=True)
    if (out / "DONE.json").is_file() and (out / "best.pth").is_file() and (out / "metrics.json").is_file():
        print(f"[skip] {approach}/{model_name} already done")
        return json.loads((out / "metrics.json").read_text())
    batch_size = BATCH_SIZE
    if (out / "last.pth").is_file():
        batch_size = min(batch_size, torch.load(out / "last.pth", map_location="cpu", weights_only=False)["batch_size"])
    while True:
        print(f"[train] {approach}/{model_name} batch={batch_size}")
        try:
            return _train_and_test(approach, model_name, out, batch_size)
        except Exception as e:
            if not is_oom(e):
                raise
            free_memory()
            if batch_size // 2 < MIN_BATCH_SIZE:
                raise
            batch_size //= 2
            print(f"  OOM -> retrying from last.pth with batch {batch_size}")

In [ ]:
all_metrics = []
for approach in APPROACHES:
    for model_name in MODEL_NAMES:
        all_metrics.append(run(approach, model_name))

In [ ]:
cols = ["approach", "model", "test_accuracy", "test_macro_f1", "test_weighted_f1",
        "test_macro_precision", "test_macro_recall", "val_accuracy", "val_macro_f1",
        "best_epoch", "epochs_run", "batch_size", "train_images"]
board = pd.DataFrame(all_metrics)[cols]
board = board.sort_values(["approach", "test_accuracy", "test_macro_f1"],
                          ascending=[True, False, False]).reset_index(drop=True)
board.insert(2, "rank", board.groupby("approach").cumcount() + 1)
board.to_csv(RESULTS_DIR / "leaderboard.csv", index=False)

lines = ["FINAL RESULTS — 10-class eye disease baselines", "=" * 72,
         f"val images (original): {len(val_df)}   test images (original): {len(test_df)}",
         "ranking within each approach: test accuracy, then test macro-F1", ""]
final = {"ranking": "test_accuracy desc, then test_macro_f1 desc, within approach", "approaches": {}}
for approach in APPROACHES:
    sub = board[board["approach"] == approach]
    lines.append(f"[{approach}]  train images: {int(sub['train_images'].iloc[0])}")
    lines.append(sub.drop(columns=["approach"]).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    top2 = sub.head(2)
    lines.append("TOP-2: " + ", ".join(
        f"#{r['rank']} {r['model']} (acc {r['test_accuracy']:.4f}, macro-F1 {r['test_macro_f1']:.4f})"
        for _, r in top2.iterrows()))
    lines.append("")
    final["approaches"][approach] = {
        "leaderboard": sub.drop(columns=["approach"]).to_dict(orient="records"),
        "top2": top2["model"].tolist(),
    }

text = "\n".join(lines) + "\n"
(RESULTS_DIR / "FINAL_RESULTS.txt").write_text(text)
(RESULTS_DIR / "FINAL_RESULTS.json").write_text(json.dumps(final, indent=2))
print(text)